# Assignment 1B — Google Colab T4 (CUDA) Execution Notebook
## Enterprise Domain-Specific LLM — V5: IT Helpdesk & Incident Resolution Assistant

This notebook executes all CUDA-dependent components of **Assignment 1B** with **real empirical measurements** (zero projected or fabricated values):
- **Part B2:** True 4-bit NF4 QLoRA fine-tuning (`BitsAndBytesConfig` + `PEFT` Adapter B: `r=16, alpha=32, q_proj, v_proj` + `TRL SFTTrainer`)
- **Part B3:** Fine-tuned adapter evaluation & evidence-backed comparative analysis
- **Part C2:** Speculative Decoding (`Qwen/Qwen2.5-1.5B-Instruct` target + `Qwen/Qwen2.5-0.5B-Instruct` draft) on CUDA
- **Part C3:** Empirical 10-prompt benchmark for `bfloat16/fp16` baseline, `4-bit NF4`, and `4-bit NF4 + Speculative Decoding` with exact `torch.cuda.max_memory_allocated()` VRAM & INR cost calculation
- **Optional Extension:** Fine-tuned Adapter + 4-bit NF4 + Speculative Decoding combined

> **Runtime requirement:** In Google Colab, select **Runtime → Change runtime type → T4 GPU** (free tier). Total runtime: ~15–20 minutes.


## 1. Environment Setup & GPU Verification


In [ ]:
!pip install -q -U "transformers>=4.46.0" "peft>=0.13.0" "trl>=0.12.0" "bitsandbytes>=0.44.0" "accelerate>=1.0.0" "datasets>=3.0.0" "matplotlib" "psutil"

import torch, platform
assert torch.cuda.is_available(), "CUDA GPU required! Switch Runtime -> T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} | Total VRAM: {vram_gb:.2f} GB | BF16 supported: {torch.cuda.is_bf16_supported()}")


## 2. Load Cleaned Instruction Dataset (`train.jsonl` & `eval.jsonl`)
Upload `train.jsonl` and `eval.jsonl` from `enterprise-domain-llm/data/` (or clone your private repo).


In [ ]:
from pathlib import Path
import json

Path("data").mkdir(exist_ok=True)
Path("results/training").mkdir(parents=True, exist_ok=True)
Path("results/evaluation").mkdir(parents=True, exist_ok=True)
Path("results/speculative").mkdir(parents=True, exist_ok=True)
Path("results/quantization").mkdir(parents=True, exist_ok=True)
Path("models/lora_adapter").mkdir(parents=True, exist_ok=True)

if not (Path("data/train.jsonl").exists() and Path("data/eval.jsonl").exists()):
    from google.colab import files
    print("Please upload train.jsonl and eval.jsonl from enterprise-domain-llm/data/:")
    uploaded = files.upload()
    for name, content in uploaded.items():
        Path("data", name).write_bytes(content)

n_train = sum(1 for _ in open("data/train.jsonl"))
n_eval = sum(1 for _ in open("data/eval.jsonl"))
print(f"Loaded dataset: {n_train} train / {n_eval} eval ({n_train + n_eval} total)")


## 3. Shared Configuration & Canonical Prompts


In [ ]:
import gc, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

TARGET_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DRAFT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
HOURLY_RATES = {"T4": 3.50, "A100": 12.00}

CANONICAL_PROMPTS = [
    {"id": "prompt_1", "category": "linux_storage", "prompt": "How do I troubleshoot an Ubuntu server that cannot mount an NTFS drive?"},
    {"id": "prompt_2", "category": "linux_performance", "prompt": "How do I investigate high disk I/O and CPU usage on a Linux production server?"},
    {"id": "prompt_3", "category": "cloud_networking", "prompt": "How do I troubleshoot AWS EC2 instance connectivity and SSH access issues?"},
]

PRODUCTION_10_PROMPTS = [
    {"id": "p1", "text": "How do I troubleshoot an Ubuntu server that cannot mount an NTFS drive?"},
    {"id": "p2", "text": "How do I investigate high disk I/O and CPU usage on a Linux production server?"},
    {"id": "p3", "text": "How do I troubleshoot AWS EC2 instance connectivity and SSH access issues?"},
    {"id": "p4", "text": "What steps should I follow when a systemd service enters a crash-loop state on Linux?"},
    {"id": "p5", "text": "How do I resolve a corporate VPN client authentication failure on macOS or Linux?"},
    {"id": "p6", "text": "How can I diagnose DNS resolution timeouts and DNS leak issues on a Linux server?"},
    {"id": "p7", "text": "What commands are used to check and repair a corrupted ext4 or XFS filesystem?"},
    {"id": "p8", "text": "How do I configure and verify Red Hat Enterprise Linux firewall rules using firewalld?"},
    {"id": "p9", "text": "How do I troubleshoot high connection pooling exhaustion in PostgreSQL database server?"},
    {"id": "p10", "text": "What is the procedure to expand an AWS EBS root volume without rebooting the instance?"},
]

def cost_per_1m(tps, rate):
    return round((1_000_000.0 / tps) / 3600.0 * rate, 2) if tps > 0 else 0.0

def clear_sampling(model):
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Compute dtype:", COMPUTE_DTYPE)


## 4. Part B2 — True 4-bit NF4 QLoRA Fine-Tuning (Adapter B: `r=16, alpha=32, q_proj, v_proj`)


In [ ]:
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

base_q = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, quantization_config=bnb_config, device_map="auto"
)
base_q.config.use_cache = False
base_q = prepare_model_for_kbit_training(base_q)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
peft_model = get_peft_model(base_q, lora_cfg)
peft_model.print_trainable_parameters()

def apply_chat(ex):
    msgs = [{"role": "user", "content": ex["instruction"]}, {"role": "assistant", "content": ex["response"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

train_ds = load_dataset("json", data_files="data/train.jsonl", split="train").map(apply_chat)
eval_ds = load_dataset("json", data_files="data/eval.jsonl", split="train").map(apply_chat)

sft_args = SFTConfig(
    output_dir="results/training/checkpoints",
    dataset_text_field="text",
    max_length=384,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=peft_model, args=sft_args,
    train_dataset=train_ds, eval_dataset=eval_ds, processing_class=tokenizer,
)

init_eval = trainer.evaluate()
print("Step 0 Eval Loss:", round(init_eval["eval_loss"], 4))
train_res = trainer.train()

peft_model.save_pretrained("models/lora_adapter")
tokenizer.save_pretrained("models/lora_adapter")

history = [{"step": 0, "epoch": 0.0, "eval_loss": round(init_eval["eval_loss"], 4)}] + list(trainer.state.log_history)
Path("results/training/training_history.json").write_text(json.dumps(history, indent=2))
print("QLoRA Training complete! Saved to models/lora_adapter/")

del trainer, peft_model, base_q
gc.collect(); torch.cuda.empty_cache()


## 5. Part B3 — Evaluate Fine-Tuned QLoRA Adapter on 3 Canonical Prompts


In [ ]:
from peft import PeftModel

eval_base = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, torch_dtype=COMPUTE_DTYPE, device_map="auto")
eval_model = PeftModel.from_pretrained(eval_base, "models/lora_adapter")
eval_model = eval_model.merge_and_unload()  # Merge LoRA for zero-overhead inference
eval_model.eval()
clear_sampling(eval_model)

adapter_runs = []
for item in CANONICAL_PROMPTS:
    inp = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = eval_model.generate(**inp, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    gen = out[0][inp["input_ids"].shape[1]:]
    text = tokenizer.decode(gen, skip_special_tokens=True).strip()
    tps = round(len(gen) / dt, 2)
    adapter_runs.append({
        "id": item["id"], "category": item["category"], "prompt": item["prompt"],
        "generated_text": text, "generated_tokens": len(gen),
        "generation_time_sec": round(dt, 3), "tokens_per_second": tps,
    })
    print(f"[{item['id']}] {tps} tok/s ({dt:.2f}s)\n{text[:240]}...\n")

Path("results/evaluation/adapter_outputs.json").write_text(json.dumps({"adapter_runs": adapter_runs}, indent=2))
del eval_model, eval_base
gc.collect(); torch.cuda.empty_cache()


## 6. Part C2 — Speculative Decoding Benchmark on CUDA (3 Canonical Prompts)


In [ ]:
target_m = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, torch_dtype=COMPUTE_DTYPE, device_map="auto").eval()
draft_m = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL, torch_dtype=COMPUTE_DTYPE, device_map="auto").eval()
clear_sampling(target_m); clear_sampling(draft_m)

# Warmup
d_in = tokenizer("Linux troubleshooting", return_tensors="pt").to("cuda")
with torch.no_grad():
    _ = target_m.generate(**d_in, max_new_tokens=10, do_sample=False)
    _ = target_m.generate(**d_in, assistant_model=draft_m, max_new_tokens=10, do_sample=False)
torch.cuda.synchronize()

spec_runs = []
for item in CANONICAL_PROMPTS:
    inp = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    in_len = inp["input_ids"].shape[1]

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        o_std = target_m.generate(**inp, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize(); dt_std = time.perf_counter() - t0
    ids_std = o_std[0][in_len:].tolist()
    tps_std = round(len(ids_std) / dt_std, 2)

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        o_spec = target_m.generate(**inp, assistant_model=draft_m, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize(); dt_spec = time.perf_counter() - t0
    ids_spec = o_spec[0][in_len:].tolist()
    tps_spec = round(len(ids_spec) / dt_spec, 2)

    spd = round(((tps_spec - tps_std) / tps_std) * 100, 1)
    prefix = sum(1 for a, b in zip(ids_std, ids_spec) if a == b)
    spec_runs.append({
        "prompt_id": item["id"], "category": item["category"], "prompt": item["prompt"],
        "standard": {"latency_sec": round(dt_std, 3), "tokens": len(ids_std), "tok_per_sec": tps_std, "text": tokenizer.decode(ids_std, skip_special_tokens=True).strip()},
        "speculative": {"latency_sec": round(dt_spec, 3), "tokens": len(ids_spec), "tok_per_sec": tps_spec, "text": tokenizer.decode(ids_spec, skip_special_tokens=True).strip()},
        "speedup_percentage": spd,
        "token_agreement": {"exact_sequence_match": ids_std == ids_spec, "common_prefix_tokens": prefix},
    })
    print(f"[{item['id']}] Standard: {tps_std} tok/s | Speculative: {tps_spec} tok/s | Speedup: {spd:+0.1f}% | Exact match: {ids_std == ids_spec}")

avg_spd = round(sum(r["speedup_percentage"] for r in spec_runs) / len(spec_runs), 1)
Path("results/speculative/speculative_results.json").write_text(json.dumps({
    "metadata": {"target_model": TARGET_MODEL, "draft_model": DRAFT_MODEL, "device": "cuda", "precision": str(COMPUTE_DTYPE), "max_new_tokens": 150, "average_speedup_pct": avg_spd},
    "runs": spec_runs,
}, indent=2))
del target_m, draft_m
gc.collect(); torch.cuda.empty_cache()


## 7. Part C3 — Empirical 4-bit NF4 Quantization & Production Cost Benchmark (10 Prompts)


In [ ]:
def bench_10(model, assistant=None):
    torch.cuda.reset_peak_memory_stats()
    tot_tok, tot_t, runs = 0, 0.0, []
    kw = {"max_new_tokens": 150, "do_sample": False, "pad_token_id": tokenizer.pad_token_id}
    if assistant is not None:
        kw["assistant_model"] = assistant
    for p in PRODUCTION_10_PROMPTS:
        inp = tokenizer(p["text"], return_tensors="pt").to("cuda")
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inp, **kw)
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        gen = len(out[0][inp["input_ids"].shape[1]:])
        tot_tok += gen; tot_t += dt
        runs.append({"id": p["id"], "generated_tokens": gen, "time_sec": round(dt, 3), "tok_per_sec": round(gen / dt, 2)})
    peak_gb = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
    avg_tps = round(tot_tok / tot_t, 2)
    return avg_tps, peak_gb, runs

configs = []

# 1. Baseline (BF16 / FP16)
m1 = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, torch_dtype=COMPUTE_DTYPE, device_map="auto").eval()
clear_sampling(m1)
_ = m1.generate(**tokenizer("Warmup", return_tensors="pt").to("cuda"), max_new_tokens=10, do_sample=False)
tps1, vram1, r1 = bench_10(m1)
configs.append({"configuration": "bfloat16 (Part A baseline)", "device": "cuda", "measured": True, "peak_vram_gb": vram1, "throughput_tok_s": tps1, "cost_t4_inr": cost_per_1m(tps1, 3.5), "cost_a100_inr": cost_per_1m(tps1, 12.0), "runs": r1})
del m1; gc.collect(); torch.cuda.empty_cache()

# 2. 4-bit NF4 Quantized
m2 = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, quantization_config=bnb_config, device_map="auto").eval()
clear_sampling(m2)
_ = m2.generate(**tokenizer("Warmup", return_tensors="pt").to("cuda"), max_new_tokens=10, do_sample=False)
tps2, vram2, r2 = bench_10(m2)
configs.append({"configuration": "4-bit NF4 quantized", "device": "cuda", "measured": True, "peak_vram_gb": vram2, "throughput_tok_s": tps2, "cost_t4_inr": cost_per_1m(tps2, 3.5), "cost_a100_inr": cost_per_1m(tps2, 12.0), "runs": r2})

# 3. 4-bit NF4 + Speculative Decoding
d2 = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL, quantization_config=bnb_config, device_map="auto").eval()
clear_sampling(d2)
_ = m2.generate(**tokenizer("Warmup", return_tensors="pt").to("cuda"), assistant_model=d2, max_new_tokens=10, do_sample=False)
tps3, vram3, r3 = bench_10(m2, assistant=d2)
configs.append({"configuration": "4-bit + Speculative Decoding (Step 2)", "device": "cuda", "measured": True, "peak_vram_gb": vram3, "throughput_tok_s": tps3, "cost_t4_inr": cost_per_1m(tps3, 3.5), "cost_a100_inr": cost_per_1m(tps3, 12.0), "runs": r3})

# 4. Optional Extension: Fine-tuned Adapter + 4-bit NF4 + Speculative
m_ext = PeftModel.from_pretrained(m2, "models/lora_adapter").eval()
tps4, vram4, r4 = bench_10(m_ext, assistant=d2)
opt_ext = {"configuration": "Fine-Tuned Adapter + 4-bit NF4 + Speculative (Optional Extension)", "device": "cuda", "measured": True, "peak_vram_gb": vram4, "throughput_tok_s": tps4, "cost_t4_inr": cost_per_1m(tps4, 3.5), "cost_a100_inr": cost_per_1m(tps4, 12.0), "runs": r4}

for c in configs + [opt_ext]:
    print(f"{c['configuration']:48s} | VRAM: {c['peak_vram_gb']:5.2f} GB | {c['throughput_tok_s']:6.2f} tok/s | T4: ₹{c['cost_t4_inr']:6.2f} | A100: ₹{c['cost_a100_inr']:6.2f}")

best = min(configs, key=lambda x: x["cost_t4_inr"])
rec = (
    f"Based on empirical CUDA T4 measurements across 10 IT Helpdesk prompts, we recommend deploying "
    f"**{best['configuration']}** on an NVIDIA T4 GPU instance at **₹{best['cost_t4_inr']} per 1M tokens** "
    f"({best['throughput_tok_s']} tok/s) using **{best['peak_vram_gb']} GB VRAM** (compared to ₹{configs[0]['cost_t4_inr']}/1M "
    f"and {configs[0]['peak_vram_gb']} GB for the unquantized baseline). This configuration delivers substantial VRAM savings "
    f"while preserving deterministic greedy command accuracy."
)
Path("results/quantization/quantization_cost_results.json").write_text(json.dumps({
    "metadata": {"model_name": TARGET_MODEL, "draft_model_name": DRAFT_MODEL, "device": "cuda", "gpu_name": torch.cuda.get_device_name(0), "base_dtype": str(COMPUTE_DTYPE), "prompts_count": 10, "max_new_tokens": 150, "hourly_rates_inr": HOURLY_RATES, "cuda_4bit_measured": True},
    "recommendation": rec,
    "configurations": configs,
    "optional_extension": opt_ext,
}, indent=2))
print("\nRecommendation:", rec)


## 8. Zip & Download All Empirical CUDA Results Back to Local Workspace


In [ ]:
import shutil
shutil.make_archive("assignment_1B_colab_results", "zip", ".", "results")
shutil.make_archive("assignment_1B_colab_adapter", "zip", ".", "models/lora_adapter")
from google.colab import files
files.download("assignment_1B_colab_results.zip")
files.download("assignment_1B_colab_adapter.zip")
print("Ready! Unzip into enterprise-domain-llm/ and re-run charts + master notebook export.")
